# Differentiating a coupled run: the response to an SST bump, and the gradient with respect to a parameter

This notebook differentiates a coupled aquaplanet run two ways. Part 1 uses forward mode (`jax.jvp`) to ask how the final sea surface temperature and surface meridional wind respond to a smooth bump in the *initial* sea surface temperature. Part 2 uses reverse mode (`jax.grad`) to ask how a scalar of the run depends on a physical *parameter* of the ocean, and checks the answer against a finite difference. On a laptop CPU it takes about 3 minutes, nearly all of it compiling the differentiated coupled step.

The coupled model comes through `jem.configurations.load`, which returns the shipped, validated `aquaplanet-slab` configuration as built Python objects: what this notebook demonstrates has nothing to do with how the model was assembled, so the assembly stays one line.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

from jem import configurations, plot, read_field, replace_field

output_dir = (Path("output") / "01-03_sst_response_jvp").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Part 1: the response to an SST bump

### Build the shipped model

In [ ]:
# `seaice="none"` is the same choice as the CLI's `seaice=none`: an
# atmosphere/ocean pair, the smallest coupled model. The ice component would
# add nothing to a warm 5-day run (its ice fraction stays zero, see
# `01_aquaplanet.ipynb`). See `jem/configurations.py`.
exp = configurations.load("aquaplanet-slab", seaice="none")
coupler = exp.coupler
coupler

### Taking the gradient of the coupled model

Mathematically speaking, we are trying to assess
$$
x_{\mathrm{final}} = F_{t_0, T}(x_0)
$$
where $x_0$ and $x_{\mathrm{final}}$ are the initial and final states of the system, $t_0$ is the initial time, $T$ the length of simulation time, and $F_{t_0, T}$ the trajectory function that maps the input state from time $t = t_0$ to $t = t_0 + T$.

The following code computes the sensitivity of the trajectory function to a smooth Gaussian bump $g(x)$ in the initial sea surface temperature: a bump of height 1 K at its centre (0°N, 180°E) with a standard deviation of 10° of longitude and latitude. Let the initial condition $x_0$ be perturbed as
$$
    x'_0(x; \epsilon) = x_0(x) + \epsilon g(x)
$$
so the final state responds as
$$
x'_{\mathrm{final}} = x_{\mathrm{final}} + \epsilon \, \delta x_{\mathrm{final}} + O(\epsilon^2)
$$
and the sensitivity of the final state to $g$, noted $s_{x_{\mathrm{final}}}$, is formally defined as
$$
s_{x_{\mathrm{final}}} = \left.\frac{\partial x'_{\mathrm{final}} }{\partial \epsilon}\right|_{\epsilon = 0} = \delta x_{\mathrm{final}}
$$

The entire computation is achieved with one forward-mode `jax.jvp` call, which returns $s_{x_{\mathrm{final}}}$ (the Jacobian of $F$ applied to $g$) without ever forming the Jacobian.

In [ ]:
forecast_days = 5
trajectory = coupler.generate_trajectory_function(forecast_days)
initial_carry = coupler.initialize()


@jax.jit
def forecast(sst):
    """Run the coupled model from an ocean initialised with `sst`.

    The carry is rebuilt here rather than mutated because `sst` is
    the argument being differentiated, so it has to reach the
    trajectory function through the carry it is called with.
    """
    carry = replace_field(initial_carry, "ocn.state.sea_surface_temperature", sst)
    final, _ = trajectory(carry)
    return (read_field(final, "ocn.state.sea_surface_temperature"),
            read_field(final, "atm.derived.v0"))

In [ ]:
sst_initial = read_field(initial_carry, "ocn.state.sea_surface_temperature")
grid = coupler.components["ocn"].grid

# The perturbation g(x): a Gaussian bump, 1 K high at its centre, centred on
# 0°N, 180°E, with a standard deviation of 10 degrees in longitude and in
# latitude. The longitude difference is wrapped to [-180, 180) so the bump is
# the same shape wherever its centre is placed relative to the grid's seam.
bump_lon, bump_lat, bump_width = 180.0, 0.0, 10.0  # degrees
lon_degrees = jnp.degrees(grid.longitude_axis_radian)
lat_degrees = jnp.degrees(grid.latitude_axis_radian)
lon_offset = (lon_degrees - bump_lon + 180.0) % 360.0 - 180.0
tangent = jnp.exp(
    -0.5 * (lon_offset[:, None] ** 2 + (lat_degrees[None, :] - bump_lat) ** 2)
    / bump_width**2
)

# jax.jvp gives the sensitivity of the final SST and surface meridional wind
# to that bump.
(sst_final, v_final), (d_sst, d_v) = jax.jvp(forecast, (sst_initial,), (tangent,))

# Check the tangent against a central finite difference along the same
# direction: the two should agree closely wherever the response is not
# noise.
step = 0.05  # K
sst_plus, v_plus = forecast(sst_initial + step * tangent)
sst_minus, v_minus = forecast(sst_initial - step * tangent)
fd_sst = (sst_plus - sst_minus) / (2 * step)
fd_v = (v_plus - v_minus) / (2 * step)
print("correlation of jvp and finite difference, d_sst:",
      float(np.corrcoef(np.ravel(d_sst), np.ravel(fd_sst))[0, 1]))
print("correlation of jvp and finite difference, d_v:  ",
      float(np.corrcoef(np.ravel(d_v), np.ravel(fd_v))[0, 1]))

### Visualization

Panels (a) and (c) are the initial and final SST, (b) is the bump, and (d)-(f) the responses (SST and meridional wind). The response panels use diverging colours with the contour levels chosen from each field: they are symmetric about zero, with an odd number of bands so that zero is the *centre* of the middle (white) band rather than a boundary. Values within one band of zero are therefore drawn white, instead of the sign of near-zero noise being drawn as signal.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

# All fields share the ocean's grid (aquaplanet: same grid as the
# atmosphere), so one pair of degree coordinates covers them all.
lon = xr.DataArray(np.degrees(grid.longitude_axis_radian), dims="lon")
lat = xr.DataArray(np.degrees(grid.latitude_axis_radian), dims="lat")


def as_field(array):
    return xr.DataArray(array, dims=("lon", "lat"), coords={"lon": lon, "lat": lat})


def symmetric_levels(field, bands=11):
    """Band edges symmetric about zero, with zero at the centre of a band.

    An odd number of bands puts zero in the middle of the central band, so a
    diverging colormap shows it as white and anything smaller than half a band
    is drawn as "no response".
    """
    peak = float(np.abs(np.asarray(field)).max())
    return np.linspace(-peak, peak, bands + 1)


sst_initial_c = np.asarray(sst_initial) - 273.15
sst_final_c = np.asarray(sst_final) - 273.15
sst_levels = np.linspace(
    min(sst_initial_c.min(), sst_final_c.min()),
    max(sst_initial_c.max(), sst_final_c.max()),
    11,
)

fig, ax = plt.subplots(3, 2, figsize=(12, 16))

plot.map_plot(as_field(sst_initial_c), ax=ax[0, 0], levels=sst_levels,
              title="(a) $\\mathrm{SST}_\\mathrm{init}$ [°C]")
plot.map_plot(as_field(tangent), ax=ax[0, 1], levels=np.linspace(0, 1, 11),
              cmap="Reds",
              title="(b) the bump $g$ [K]")
plot.map_plot(as_field(sst_final_c), ax=ax[1, 0], levels=sst_levels,
              title="(c) $\\mathrm{SST}_\\mathrm{final}$ [°C]")
plot.map_plot(as_field(d_sst), ax=ax[1, 1], levels=symmetric_levels(d_sst),
              cmap="bwr",
              title="(d) $\\partial \\mathrm{SST}_\\mathrm{final}$ [K per K of bump]")
plot.map_plot(as_field(v_final), ax=ax[2, 0], levels=symmetric_levels(v_final),
              cmap="bwr",
              title="(e) $\\mathrm{v}_\\mathrm{final}$ [m/s]")
plot.map_plot(as_field(d_v), ax=ax[2, 1], levels=symmetric_levels(d_v),
              cmap="bwr",
              title="(f) $\\partial \\mathrm{v}_\\mathrm{final}$ [m/s per K of bump]")

fig.suptitle(f"Response time: {forecast_days:.1f} days")
for _ax in ax.flatten():
    _ax.set_xlabel("longitude [deg]")
    _ax.set_ylabel("latitude [deg]")
plt.tight_layout()

In [ ]:
fig.savefig(output_dir / "sensitivity.png", dpi=150)

## Part 2: the gradient with respect to a physical parameter

The slab ocean's numeric parameters travel in the carry (`carry.components["ocn"]["params"]`) as differentiable leaves -- see *Parameters: process and initial condition* in `docs/source/python_api.md` -- so `jax.grad` differentiates a coupled run with respect to a parameter exactly as `jax.jvp` did with respect to a field. The parameter here is `mixed_layer_depth_min`, the depth (m) of the ocean's mixed layer at the equator (it deepens poleward to `mixed_layer_depth_max`). It is a *process* parameter, read from the carry on every step, so replacing that leaf in the initial carry is all it takes.

The scalar is the area-weighted mean SST after a 2-day run, and a central finite difference over the same run checks the gradient.

In [ ]:
import dataclasses

parameter_days = 2
parameter_trajectory = coupler.generate_trajectory_function(parameter_days)
latitude_weights = jnp.cos(jnp.asarray(grid.latitude_axis_radian))
latitude_weights = latitude_weights / latitude_weights.sum()


def mean_sst_after_run(mixed_layer_depth_min):
    """Return the area-mean SST [K] after the run, for one ocean parameter."""
    ocean = initial_carry.components["ocn"]
    ocean = dict(ocean, params=ocean["params"].replace(
        mixed_layer_depth_min=mixed_layer_depth_min))
    carry = dataclasses.replace(
        initial_carry, components=dict(initial_carry.components, ocn=ocean))
    final, _ = parameter_trajectory(carry)
    sst = read_field(final, "ocn.state.sea_surface_temperature")
    return (sst.mean(axis=0) * latitude_weights).sum()


depth = jnp.float32(initial_carry.components["ocn"]["params"].mixed_layer_depth_min)
mean_sst, gradient = jax.jit(jax.value_and_grad(mean_sst_after_run))(depth)

# A central finite difference: two more runs, the parameter moved by +-5 %.
run = jax.jit(mean_sst_after_run)
delta = 0.05 * depth
finite_difference = (run(depth + delta) - run(depth - delta)) / (2 * delta)

mean_sst_initial = (sst_initial.mean(axis=0) * latitude_weights).sum()
print(f"mixed_layer_depth_min = {float(depth):.1f} m")
print(f"area-mean SST: {float(mean_sst_initial):.3f} K initially, "
      f"{float(mean_sst):.3f} K after {parameter_days} days")
print(f"jax.grad:          d(mean SST)/d(depth) = {float(gradient):.5f} K/m")
print(f"finite difference: d(mean SST)/d(depth) = {float(finite_difference):.5f} K/m")

# The finite difference is itself the difference of two float32 runs, so it is
# only trusted to a few percent: the check is a 5 % relative tolerance.
assert jnp.isfinite(gradient) and gradient != 0.0
assert abs(gradient - finite_difference) <= 0.05 * abs(finite_difference)

The number is the change in the area-mean sea surface temperature after 2 days (K) per metre of equatorial mixed-layer depth. Over these 2 days the ocean cools slightly (the two means printed above), and a deeper mixed layer has a larger heat capacity, so the same surface heat flux changes its temperature less: the gradient is positive. `jax.grad` obtained it from one forward and one backward pass through the whole coupled run -- atmosphere, exchange and ocean -- and it agrees with the two-run finite difference to within the stated 5 % tolerance. With many parameters the cost of the gradient would not grow, where a finite difference needs two runs per parameter.